In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
def assign_label(row):
    if row["strong_snr_candidate"]:
        return "SNR"

    if row["clean_hii"]:
        return "HII"
    
    if row["dig_seed_region"]:
        return "DIG"    

    if row["possible_hii"]:
        return "HII_possible"

    if row["possible_snr_candidate"]:
        return "SNR_possible"

    return "uncertain"

In [4]:
regions = pd.read_csv("HII_CAT.csv")
pn = pd.read_csv("PN_CAT.csv")

print("Loaded regions:", len(regions))
print("Loaded PNe:", len(pn))

Loaded regions: 425
Loaded PNe: 571


In [5]:
min_SNR = 3.0
SNR_thresh = 0.40      
DIG_thresh = 0.30
M33_HII_thresh = 0.35

In [6]:
regions["ha_surface"] = regions["F_ha"] / regions["npix"]

regions["good_ha_sii"] = ((regions["sn_ha"] >= min_SNR) & (regions["sn_sii"] >= min_SNR) & np.isfinite(regions["sii_ha"]))

regions["strong_snr_candidate"] = (regions["good_ha_sii"] & (regions["sii_ha"] >= SNR_thresh))

regions["possible_snr_candidate"] = (regions["good_ha_sii"] & (regions["sii_ha"] >= 0.35) & (regions["sii_ha"] < SNR_thresh))

regions["clean_hii"] =((regions["bpt_class"] == "HII_BPT_clean") &regions["good_ha_sii"] &
    (regions["sii_ha"] < M33_HII_thresh) &(~regions["strong_snr_candidate"]))

regions["possible_hii"] = ((regions["bpt_class"].isin(["HII_BPT_clean", "HII_BPT_possible"])) &
    regions["good_ha_sii"] &(regions["sii_ha"] < SNR_thresh) &
    (~regions["clean_hii"]) &(~regions["strong_snr_candidate"]))

In [7]:
regions["dig_seed_region"] = (regions["good_ha_sii"] &(regions["sii_ha"] >= DIG_thresh) &(~regions["strong_snr_candidate"]) &
    (~regions["clean_hii"]))

In [8]:
regions["simple_class"] = regions.apply(assign_label, axis=1)
regions.to_csv("all_regions_labeled.csv", index=False)

In [9]:
print(regions["simple_class"].value_counts())

simple_class
HII             256
DIG              86
SNR              80
HII_possible      3
Name: count, dtype: int64
